In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         COMPLETE NEWS ANALYTICS SYSTEM — WITH BERTOPIC TOPIC MODELING        ║
║          RSS Collection + DistilBERT Classification + BERTopic              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
import re
import time
import warnings
import random
from collections import Counter
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# RSS fetching
import feedparser
from bs4 import BeautifulSoup
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ML libraries
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim import AdamW

# Transformers
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

# BERTopic for topic modeling
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Data collection
    MIN_ARTICLES_PER_CLASS = 15
    MAX_ARTICLES_PER_FEED = 50
    FETCH_TIMEOUT = 15
    RETRY_COUNT = 3
    
    # Model
    MODEL_NAME = "distilbert-base-uncased"
    MAX_LEN = 256
    BATCH_SIZE = 4
    EPOCHS = 5
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # Training
    VALIDATION_SPLIT = 0.15
    TEST_SPLIT = 0.15
    GRADIENT_ACCUMULATION_STEPS = 2
    MAX_GRAD_NORM = 1.0
    
    # Augmentation
    AUGMENTATION_FACTOR = 2
    
    # BERTopic
    N_TOPICS = 10
    MIN_TOPIC_SIZE = 5
    
    # Output
    OUTPUT_DIR = "output_complete"
    SAVE_MODEL = True
    
    # Categories
    CATEGORIES = ["World", "Business", "Technology", "Science", "Sports", "Entertainment"]
    
    # RSS Feeds
    RSS_FEEDS = [
        ("http://feeds.bbci.co.uk/news/world/rss.xml", "World"),
        ("http://feeds.bbci.co.uk/news/business/rss.xml", "Business"),
        ("http://feeds.bbci.co.uk/news/technology/rss.xml", "Technology"),
        ("http://feeds.bbci.co.uk/news/science_and_environment/rss.xml", "Science"),
        ("http://feeds.bbci.co.uk/sport/rss.xml", "Sports"),
        ("http://feeds.bbci.co.uk/news/entertainment_and_arts/rss.xml", "Entertainment"),
        ("https://www.aljazeera.com/xml/rss/all.xml", "World"),
        ("https://techcrunch.com/feed/", "Technology"),
    ]

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ============================================================================
# DATA AUGMENTATION
# ============================================================================

class DataAugmenter:
    @staticmethod
    def synonym_replacement(text: str, n: int = 1) -> str:
        words = text.split()
        if len(words) < 5:
            return text
        
        synonyms = {
            'good': ['great', 'excellent', 'positive'],
            'bad': ['poor', 'negative', 'terrible'],
            'big': ['large', 'huge', 'massive'],
            'small': ['tiny', 'little', 'minor'],
            'important': ['significant', 'crucial', 'vital'],
            'new': ['novel', 'fresh', 'recent'],
            'old': ['ancient', 'aged', 'former'],
            'many': ['numerous', 'countless', 'multiple'],
            'said': ['stated', 'reported', 'announced'],
            'made': ['created', 'produced', 'developed'],
        }
        
        words_copy = words.copy()
        for _ in range(min(n, len(words_copy) // 2)):
            idx = random.randint(0, len(words_copy) - 1)
            word = words_copy[idx].lower()
            if word in synonyms:
                words_copy[idx] = random.choice(synonyms[word])
        
        return ' '.join(words_copy)
    
    @staticmethod
    def augment_dataset(df: pd.DataFrame, factor: int = 2) -> pd.DataFrame:
        if factor <= 1:
            return df
        
        augmented_dfs = [df]
        
        for i in range(factor - 1):
            df_aug = df.copy()
            df_aug['text'] = df_aug['text'].apply(lambda x: DataAugmenter.synonym_replacement(x, n=1))
            df_aug['title'] = df_aug['title'] + f"_aug{i}"
            augmented_dfs.append(df_aug)
        
        result = pd.concat(augmented_dfs, ignore_index=True)
        print(f"  Augmented data: {len(df)} → {len(result)} articles")
        return result

# ============================================================================
# DATA COLLECTION
# ============================================================================

class RSSCollector:
    def __init__(self):
        self.session = self._create_session()
        
    def _create_session(self):
        session = requests.Session()
        retry = Retry(total=cfg.RETRY_COUNT, backoff_factor=0.5, status_forcelist=[500, 502, 503, 504])
        adapter = HTTPAdapter(max_retries=retry)
        session.mount('http://', adapter)
        session.mount('https://', adapter)
        session.headers.update({'User-Agent': 'Mozilla/5.0 (compatible; NewsAnalyzer/1.0)'})
        return session
    
    def strip_html(self, text: str) -> str:
        if not text:
            return ""
        try:
            return BeautifulSoup(text, "html.parser").get_text(separator=" ")
        except:
            return re.sub(r"<[^>]+>", " ", text)
    
    def fetch_feed(self, url: str, category: str) -> List[Dict]:
        articles = []
        try:
            response = self.session.get(url, timeout=cfg.FETCH_TIMEOUT)
            response.raise_for_status()
            feed = feedparser.parse(response.content)
            
            for entry in feed.entries[:cfg.MAX_ARTICLES_PER_FEED]:
                title = self.strip_html(entry.get("title", ""))
                summary = self.strip_html(entry.get("summary", ""))
                description = self.strip_html(entry.get("description", ""))
                
                content = summary if len(summary) > len(description) else description
                full_text = f"{title}. {content}".strip()
                
                if len(full_text) > 100 and len(full_text.split()) > 20:
                    articles.append({
                        "title": title[:100],
                        "text": full_text,
                        "category": category,
                    })
        except Exception as e:
            pass
        return articles
    
    def collect(self) -> pd.DataFrame:
        print("\n📡 Fetching RSS feeds...\n")
        all_articles = []
        
        for url, category in cfg.RSS_FEEDS:
            articles = self.fetch_feed(url, category)
            if articles:
                print(f"  ✓ {category:12} → {len(articles):2} articles")
            all_articles.extend(articles)
            time.sleep(0.2)
        
        df = pd.DataFrame(all_articles)
        
        if len(df) == 0:
            print("\n⚠️ Creating sample data...")
            df = self._create_sample_data()
        
        df = df.drop_duplicates(subset=["text"])
        df = df.reset_index(drop=True)
        
        print(f"\n📊 Total collected: {len(df)} articles")
        print(f"📊 Distribution:\n{df['category'].value_counts()}")
        
        return df
    
    def _create_sample_data(self) -> pd.DataFrame:
        samples = []
        texts = {
            "World": [
                "UN Security Council holds emergency meeting on Ukraine conflict calling for ceasefire",
                "China and Russia strengthen diplomatic ties amid global tensions",
                "European Union announces new sanctions package targeting Russian exports",
                "US President meets with NATO allies to discuss defense spending",
                "Middle East peace talks resume in Cairo with international mediators",
            ],
            "Business": [
                "Apple reports record quarterly revenue of $124 billion beating analyst expectations",
                "Microsoft stock surges after announcing new AI features across product lineup",
                "Federal Reserve signals potential rate cuts later this year",
                "Amazon expands healthcare services with new acquisition",
                "Tesla shares jump as company delivers record number of electric vehicles",
            ],
            "Technology": [
                "Google launches new Gemini AI model capable of processing multiple data types",
                "Tesla unveils next generation battery technology with 500 mile range",
                "Scientists achieve quantum computing breakthrough solving complex problems",
                "Apple announces new mixed reality headset with spatial computing",
                "Microsoft integrates ChatGPT into Bing search engine",
            ],
            "Science": [
                "NASA's James Webb telescope captures unprecedented images of distant exoplanets",
                "Researchers develop gene therapy that successfully reverses blindness",
                "New study reveals breakthrough in cancer treatment using immunotherapy",
                "Scientists discover ancient fossil revealing new dinosaur species",
                "Climate researchers warn of record temperatures due to El Nino",
            ],
            "Sports": [
                "Dodgers win World Series in dramatic Game 7 with walk-off home run",
                "LeBron James becomes first player to score 40000 career points",
                "US Women's National Team advances to World Cup final after overtime victory",
                "Manchester City wins Premier League title on final day",
                "Novak Djokovic claims record Grand Slam title at Australian Open",
            ],
            "Entertainment": [
                "New James Bond film breaks box office records with $200 million opening",
                "Taylor Swift's Eras Tour becomes highest grossing concert tour ever",
                "Netflix announces billion dollar deal for new Harry Potter series",
                "Oscar nominations announced with several records broken",
                "Disney reveals slate of upcoming Marvel movies",
            ],
        }
        
        for category, text_list in texts.items():
            for text in text_list:
                for _ in range(2):
                    samples.append({"title": text[:50], "text": text, "category": category})
        
        return pd.DataFrame(samples)

# ============================================================================
# TEXT PREPROCESSING
# ============================================================================

class TextPreprocessor:
    def clean_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'[^a-zA-Z0-9\s\.]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def preprocess_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        print("\n🔧 Preprocessing text...")
        df["clean_text"] = df["text"].apply(self.clean_text)
        df = df[df["clean_text"].str.len() > 50]
        df = df.reset_index(drop=True)
        
        text_lengths = df["clean_text"].str.split().str.len()
        print(f"  Articles after cleaning: {len(df)}")
        print(f"  Avg tokens: {text_lengths.mean():.0f} | Max: {text_lengths.max()} | Min: {text_lengths.min()}")
        
        return df

# ============================================================================
# DATASET
# ============================================================================

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long)
        }

# ============================================================================
# DISTILBERT CLASSIFIER
# ============================================================================

class DistilBertClassifier:
    def __init__(self, num_classes: int):
        self.num_classes = num_classes
        self.tokenizer = DistilBertTokenizer.from_pretrained(cfg.MODEL_NAME)
        self.model = DistilBertForSequenceClassification.from_pretrained(
            cfg.MODEL_NAME,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        self.model.to(DEVICE)
        self.label_encoder = LabelEncoder()
        
    def prepare_data(self, texts: List[str], labels: List[str]):
        y_encoded = self.label_encoder.fit_transform(labels)
        self.num_classes = len(self.label_encoder.classes_)
        
        X_train, X_temp, y_train, y_temp = train_test_split(
            texts, y_encoded, test_size=0.3, random_state=SEED, stratify=y_encoded
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
        )
        
        class_counts = Counter(y_train)
        if len(class_counts) > 1:
            class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
            sample_weights = [class_weights[y] for y in y_train]
            sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        else:
            sampler = None
        
        train_dataset = NewsDataset(X_train, y_train, self.tokenizer, cfg.MAX_LEN)
        val_dataset = NewsDataset(X_val, y_val, self.tokenizer, cfg.MAX_LEN)
        test_dataset = NewsDataset(X_test, y_test, self.tokenizer, cfg.MAX_LEN)
        
        self.train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler if sampler else None, shuffle=sampler is None, num_workers=0)
        self.val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
        self.test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
        
        print(f"\n📊 Data split: Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
        print(f"  Classes: {list(self.label_encoder.classes_)}")
        
        return self.train_loader, self.val_loader, self.test_loader
    
    def train_epoch(self, optimizer, scheduler):
        self.model.train()
        total_loss = 0
        predictions = []
        true_labels = []
        optimizer.zero_grad()
        
        for step, batch in enumerate(tqdm(self.train_loader, desc="Training")):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / cfg.GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            
            if (step + 1) % cfg.GRADIENT_ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            total_loss += loss.item() * cfg.GRADIENT_ACCUMULATION_STEPS
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
        
        avg_loss = total_loss / len(self.train_loader)
        accuracy = accuracy_score(true_labels, predictions)
        return avg_loss, accuracy
    
    def evaluate(self, loader, return_predictions=False):
        self.model.eval()
        total_loss = 0
        predictions = []
        true_labels = []
        
        with torch.no_grad():
            for batch in tqdm(loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                predictions.extend(preds.cpu().numpy())
                true_labels.extend(labels.cpu().numpy())
        
        avg_loss = total_loss / len(loader)
        accuracy = accuracy_score(true_labels, predictions)
        f1 = f1_score(true_labels, predictions, average="weighted")
        precision = precision_score(true_labels, predictions, average="weighted", zero_division=0)
        recall = recall_score(true_labels, predictions, average="weighted", zero_division=0)
        
        if return_predictions:
            return avg_loss, accuracy, f1, precision, recall, predictions, true_labels
        return avg_loss, accuracy, f1, precision, recall
    
    def train(self):
        optimizer = AdamW(self.model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
        total_steps = len(self.train_loader) * cfg.EPOCHS // cfg.GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
        
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
        best_val_f1 = 0
        
        print(f"\n🚀 Starting training for {cfg.EPOCHS} epochs...")
        
        for epoch in range(cfg.EPOCHS):
            print(f"\n{'='*50}\nEpoch {epoch + 1}/{cfg.EPOCHS}\n{'='*50}")
            train_loss, train_acc = self.train_epoch(optimizer, scheduler)
            val_loss, val_acc, val_f1, _, _ = self.evaluate(self.val_loader)
            
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["val_f1"].append(val_f1)
            
            print(f"\n📈 Results: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
            print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                if cfg.SAVE_MODEL:
                    self.save_model()
                    print(f"  ✓ Model saved (F1: {val_f1:.4f})")
        
        if cfg.SAVE_MODEL:
            self.load_model()
        
        test_loss, test_acc, test_f1, test_prec, test_rec, predictions, true_labels = self.evaluate(self.test_loader, return_predictions=True)
        print(f"\n🎯 Test Results: Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%) | F1-Score: {test_f1:.4f} ({test_f1*100:.2f}%)")
        
        return history, predictions, true_labels
    
    def save_model(self):
        model_path = os.path.join(cfg.OUTPUT_DIR, "best_model")
        self.model.save_pretrained(model_path)
        self.tokenizer.save_pretrained(model_path)
        import joblib
        joblib.dump(self.label_encoder, os.path.join(model_path, "label_encoder.pkl"))
    
    def load_model(self):
        model_path = os.path.join(cfg.OUTPUT_DIR, "best_model")
        if os.path.exists(model_path):
            self.model = DistilBertForSequenceClassification.from_pretrained(model_path)
            self.tokenizer = DistilBertTokenizer.from_pretrained(model_path)
            self.model.to(DEVICE)
            import joblib
            encoder_path = os.path.join(model_path, "label_encoder.pkl")
            if os.path.exists(encoder_path):
                self.label_encoder = joblib.load(encoder_path)
    
    def predict(self, text: str, preprocessor: TextPreprocessor) -> Dict:
        self.model.eval()
        cleaned = preprocessor.clean_text(text)
        encoding = self.tokenizer(cleaned, truncation=True, padding="max_length", max_length=cfg.MAX_LEN, return_tensors="pt")
        
        with torch.no_grad():
            input_ids = encoding["input_ids"].to(DEVICE)
            attention_mask = encoding["attention_mask"].to(DEVICE)
            outputs = self.model(input_ids, attention_mask=attention_mask)
            probs = F.softmax(outputs.logits, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()
            confidence = torch.max(probs).item()
        
        label = self.label_encoder.inverse_transform([pred_idx])[0]
        return {"prediction": label, "confidence": confidence}

# ============================================================================
# BERTOPIC TOPIC MODELING (NEW! FULL IMPLEMENTATION)
# ============================================================================

class TopicModeler:
    """BERTopic-based topic detection for emerging news themes"""
    
    def __init__(self):
        self.model = None
        self.topics_df = None
        
    def fit(self, texts: List[str], n_topics: int = 10):
        """Fit BERTopic model and extract emerging topics"""
        print("\n🔍 Training BERTopic model for emerging topic detection...")
        print(f"   Analyzing {len(texts)} documents...")
        
        try:
            # Use a lightweight sentence transformer for CPU efficiency
            sentence_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
            
            # Initialize BERTopic with custom settings for news articles
            self.model = BERTopic(
                embedding_model=sentence_model,
                nr_topics=n_topics,
                min_topic_size=3,
                verbose=True,
                calculate_probabilities=False
            )
            
            # Fit the model
            topics, probabilities = self.model.fit_transform(texts)
            
            # Get topic information
            self.topics_df = self.model.get_topic_info()
            
            # Print discovered topics
            print(f"\n📚 Discovered {len(self.topics_df) - 1} emerging topics:\n")
            print("-" * 70)
            
            for idx, row in self.topics_df.head(n_topics + 1).iterrows():
                if row["Topic"] != -1:  # Skip outlier topic
                    words = self.model.get_topic(row["Topic"])[:5]
                    word_str = " → ".join([w for w, _ in words])
                    print(f"  Topic {row['Topic']:>2}: {word_str}")
                    print(f"       ({row['Count']} articles, {row['Name']})")
                    print()
            
            return self.topics_df
            
        except Exception as e:
            print(f"⚠️ BERTopic failed: {e}")
            return None
    
    def visualize(self, output_dir: str):
        """Generate comprehensive BERTopic visualizations"""
        if self.model is None:
            print("  No BERTopic model to visualize")
            return
        
        print("\n📊 Generating BERTopic visualizations...")
        
        try:
            # 1. Topic Hierarchy (interactive)
            fig_hierarchy = self.model.visualize_hierarchy()
            fig_hierarchy.write_html(os.path.join(output_dir, "bertopic_hierarchy.html"))
            print("  ✓ Saved: bertopic_hierarchy.html")
            
            # 2. Topic Barchart
            fig_barchart = self.model.visualize_barchart(top_n_topics=cfg.N_TOPICS)
            fig_barchart.write_html(os.path.join(output_dir, "bertopic_barchart.html"))
            print("  ✓ Saved: bertopic_barchart.html")
            
            # 3. Topic Heatmap (static)
            fig_heatmap = self.model.visualize_heatmap()
            fig_heatmap.write_html(os.path.join(output_dir, "bertopic_heatmap.html"))
            print("  ✓ Saved: bertopic_heatmap.html")
            
            # 4. Static matplotlib visualization
            self._create_static_topic_chart(output_dir)
            
        except Exception as e:
            print(f"  ⚠️ Some visualizations failed: {e}")
    
    def _create_static_topic_chart(self, output_dir: str):
        """Create static matplotlib visualization of topics"""
        if self.topics_df is None:
            return
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Filter out outlier topic (-1)
        topic_data = self.topics_df[self.topics_df["Topic"] != -1].head(10)
        
        colors = plt.cm.Set3(range(len(topic_data)))
        bars = ax.barh(range(len(topic_data)), topic_data["Count"].values, color=colors)
        
        # Add topic labels
        for i, (idx, row) in enumerate(topic_data.iterrows()):
            topic_words = self.model.get_topic(row["Topic"])[:3]
            word_str = " | ".join([w for w, _ in topic_words])
            ax.text(row["Count"] + 1, i, f"Topic {row['Topic']}: {word_str}", 
                   va='center', fontsize=9)
        
        ax.set_yticks(range(len(topic_data)))
        ax.set_yticklabels([f"Topic {row['Topic']}" for _, row in topic_data.iterrows()])
        ax.set_xlabel("Number of Articles")
        ax.set_title("BERTopic: Emerging Topics in News Corpus", fontsize=14, fontweight="bold")
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "bertopic_topics.png"), dpi=150, bbox_inches="tight")
        plt.close()
        print("  ✓ Saved: bertopic_topics.png")
    
    def get_emerging_topics(self) -> pd.DataFrame:
        """Return the most significant emerging topics"""
        if self.topics_df is None:
            return pd.DataFrame()
        
        # Return topics sorted by size (excluding outlier)
        return self.topics_df[self.topics_df["Topic"] != -1].head(10)

# ============================================================================
# VISUALIZATION
# ============================================================================

class Visualizer:
    @staticmethod
    def plot_training_history(history: Dict, output_path: str):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        epochs = range(1, len(history["train_loss"]) + 1)
        
        axes[0].plot(epochs, history["train_loss"], 'b-', label='Train Loss')
        axes[0].plot(epochs, history["val_loss"], 'r-', label='Val Loss')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(epochs, history["train_acc"], 'b-', label='Train Acc')
        axes[1].plot(epochs, history["val_acc"], 'r-', label='Val Acc')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        
        axes[2].plot(epochs, history["val_f1"], 'g-', label='Val F1')
        axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1 Score'); axes[2].set_title('F1 Score')
        axes[2].legend(); axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✓ Saved: {output_path}")
    
    @staticmethod
    def plot_confusion_matrix(y_true, y_pred, labels, output_path: str):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
        plt.xlabel('Predicted', fontsize=12); plt.ylabel('Actual', fontsize=12)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✓ Saved: {output_path}")

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║      COMPLETE NEWS ANALYTICS SYSTEM — WITH BERTOPIC TOPIC MODELING           ║
║          RSS Collection + DistilBERT Classification + BERTopic              ║
╚══════════════════════════════════════════════════════════════════════════════╝
    """)
    
    # Stage 1: Data Collection
    print("\n" + "="*60)
    print("  STAGE 1: DATA COLLECTION")
    print("="*60)
    collector = RSSCollector()
    df = collector.collect()
    
    # Stage 2: Data Augmentation
    print("\n" + "="*60)
    print("  STAGE 2: DATA AUGMENTATION")
    print("="*60)
    augmenter = DataAugmenter()
    df = augmenter.augment_dataset(df, factor=cfg.AUGMENTATION_FACTOR)
    
    # Stage 3: Preprocessing
    print("\n" + "="*60)
    print("  STAGE 3: TEXT PREPROCESSING")
    print("="*60)
    preprocessor = TextPreprocessor()
    df = preprocessor.preprocess_dataframe(df)
    
    if len(df) < 20:
        print("\n❌ Insufficient data. Exiting.")
        return
    
    # Stage 4: DistilBERT Classification
    print("\n" + "="*60)
    print("  STAGE 4: DISTILBERT CLASSIFICATION")
    print("="*60)
    
    texts = df["clean_text"].tolist()
    labels = df["category"].tolist()
    
    classifier = DistilBertClassifier(num_classes=len(cfg.CATEGORIES))
    classifier.prepare_data(texts, labels)
    history, predictions, true_labels = classifier.train()
    
    # Stage 5: Visualizations
    print("\n" + "="*60)
    print("  STAGE 5: CLASSIFICATION VISUALIZATIONS")
    print("="*60)
    visualizer = Visualizer()
    visualizer.plot_training_history(history, os.path.join(cfg.OUTPUT_DIR, "training_history.png"))
    visualizer.plot_confusion_matrix(true_labels, predictions, classifier.label_encoder.classes_, 
                                    os.path.join(cfg.OUTPUT_DIR, "confusion_matrix.png"))
    
    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(true_labels, predictions, target_names=classifier.label_encoder.classes_))
    
    # Stage 6: BERTOPIC TOPIC MODELING (EMERGING TOPICS)
    print("\n" + "="*60)
    print("  STAGE 6: BERTOPIC EMERGING TOPIC DETECTION")
    print("="*60)
    
    # Use all cleaned texts for topic modeling
    topic_modeler = TopicModeler()
    topic_df = topic_modeler.fit(df["clean_text"].tolist(), n_topics=cfg.N_TOPICS)
    topic_modeler.visualize(cfg.OUTPUT_DIR)
    
    # Display emerging topics summary
    if topic_df is not None:
        print("\n📈 TOP EMERGING TOPICS (Cross-Cutting Themes):")
        print("-" * 60)
        for idx, row in topic_df.head(cfg.N_TOPICS).iterrows():
            if row["Topic"] != -1:
                words = topic_modeler.model.get_topic(row["Topic"])[:4]
                word_str = " → ".join([w for w, _ in words])
                print(f"  📰 Topic {row['Topic']}: {word_str}")
                print(f"     📄 {row['Count']} articles | {row['Name'][:50]}")
                print()
    
    # Stage 7: Sample Predictions
    print("\n" + "="*60)
    print("  STAGE 7: SAMPLE PREDICTIONS")
    print("="*60)
    
    test_samples = [
        "UN Security Council discusses Ukraine ceasefire and humanitarian corridors",
        "Apple reports record revenue with strong iPhone sales beating expectations",
        "Google announces breakthrough in quantum computing technology",
        "NASA's James Webb telescope discovers new exoplanet in habitable zone",
        "Team wins championship in dramatic overtime victory before home crowd",
        "Blockbuster movie breaks box office records during opening weekend",
    ]
    
    expected = ["World", "Business", "Technology", "Science", "Sports", "Entertainment"]
    
    print(f"\n{'Text':<45} {'Predicted':<12} {'Confidence':>10} {'Expected':<12}")
    print("-" * 85)
    
    correct = 0
    for text, exp in zip(test_samples, expected):
        result = classifier.predict(text, preprocessor)
        confidence = result["confidence"] * 100
        match = result["prediction"] == exp
        if match:
            correct += 1
        icon = "✓" if match else "✗"
        print(f"{text[:42]:<45} {result['prediction']:<12} {confidence:>9.1f}%  {icon}  {exp}")
    
    print(f"\n📊 Sample Accuracy: {correct}/{len(test_samples)} = {correct/len(test_samples)*100:.1f}%")
    
    # Final Summary
    final_accuracy = accuracy_score(true_labels, predictions)
    final_f1 = f1_score(true_labels, predictions, average="weighted")
    
    print("\n" + "="*60)
    print("  PIPELINE COMPLETE — FINAL SUMMARY")
    print("="*60)
    print(f"""
    ┌────────────────────────────────────────────────────────────────┐
    │  RESULTS                                                       │
    │    DistilBERT Classification Accuracy : {final_accuracy*100:.2f}%                       │
    │    DistilBERT F1-Score                : {final_f1*100:.2f}%                       │
    │    Test Samples                       : {len(predictions)}                               │
    │                                                                 │
    │  BERTOPIC EMERGING TOPICS                                      │
    │    Topics Discovered                  : {len(topic_df) - 1 if topic_df is not None else 0}                               │
    │                                                                 │
    │  OUTPUT FILES                                                  │
    │    classification/                                             │
    │      ├── training_history.png                                  │
    │      ├── confusion_matrix.png                                  │
    │      └── best_model/                                           │
    │                                                                 │
    │    topic_modeling/                                             │
    │      ├── bertopic_topics.png (static)                          │
    │      ├── bertopic_hierarchy.html (interactive)                 │
    │      ├── bertopic_barchart.html (interactive)                  │
    │      └── bertopic_heatmap.html (interactive)                   │
    └────────────────────────────────────────────────────────────────┘
    """)
    
    print("✅ Complete pipeline execution successful!")
    print(f"📁 All outputs saved to: {cfg.OUTPUT_DIR}/")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️ Interrupted by user")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

Using device: cpu

╔══════════════════════════════════════════════════════════════════════════════╗
║      COMPLETE NEWS ANALYTICS SYSTEM — WITH BERTOPIC TOPIC MODELING           ║
║          RSS Collection + DistilBERT Classification + BERTopic              ║
╚══════════════════════════════════════════════════════════════════════════════╝
    

  STAGE 1: DATA COLLECTION

📡 Fetching RSS feeds...

  ✓ World        → 35 articles
  ✓ Business     → 43 articles
  ✓ Technology   → 12 articles
  ✓ Science      → 35 articles
  ✓ Sports       → 48 articles
  ✓ Entertainment → 21 articles
  ✓ World        → 24 articles
  ✓ Technology   → 18 articles

📊 Total collected: 224 articles
📊 Distribution:
category
World            59
Sports           48
Business         39
Science          31
Technology       27
Entertainment    20
Name: count, dtype: int64

  STAGE 2: DATA AUGMENTATION
  Augmented data: 224 → 448 articles

  STAGE 3: TEXT PREPROCESSING

🔧 Preprocessing text...
  Articles after cleani

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5510.63it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 Data split: Train: 313 | Val: 67 | Test: 68
  Classes: [np.str_('Business'), np.str_('Entertainment'), np.str_('Science'), np.str_('Sports'), np.str_('Technology'), np.str_('World')]

🚀 Starting training for 5 epochs...

Epoch 1/5


Evaluating: 100%|██████████| 17/17 [00:03<00:00,  5.05it/s]



📈 Results: Train Loss: 1.7440 | Train Acc: 0.3291
  Val Loss: 1.5527 | Val Acc: 0.7463 | Val F1: 0.7051


...


📈 Results: Train Loss: 0.4018 | Train Acc: 0.9681
  Val Loss: 0.4920 | Val Acc: 0.9104 | Val F1: 0.9069


Evaluating: 100%|██████████| 17/17 [00:03<00:00,  4.46it/s]



🎯 Test Results: Accuracy: 0.8529 (85.29%) | F1-Score: 0.8507 (85.07%)

  STAGE 5: CLASSIFICATION VISUALIZATIONS
  ✓ Saved: output_complete/training_history.png
  ✓ Saved: output_complete/confusion_matrix.png

📊 Classification Report:
               precision    recall  f1-score   support

     Business       0.67      0.67      0.67        12
Entertainment       0.86      1.00      0.92         6
      Science       0.88      0.78      0.82         9
       Sports       0.94      1.00      0.97        15
   Technology       0.80      1.00      0.89         8
        World       0.93      0.78      0.85        18

     accuracy                           0.85        68
    macro avg       0.84      0.87      0.85        68
 weighted avg       0.86      0.85      0.85        68


  STAGE 6: BERTOPIC EMERGING TOPIC DETECTION

🔍 Training BERTopic model for emerging topic detection...
   Analyzing 448 documents...


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 11907.64it/s]
2026-04-25 19:20:15,944 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 14/14 [00:01<00:00,  8.92it/s]
2026-04-25 19:20:17,525 - BERTopic - Embedding - Completed ✓
2026-04-25 19:20:17,526 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-25 19:20:25,704 - BERTopic - Dimensionality - Completed ✓
2026-04-25 19:20:25,705 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-25 19:20:25,726 - BERTopic - Cluster - Completed ✓
2026-04-25 19:20:25,726 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-25 19:20:25,789 - BERTopic - Representation - Completed ✓
2026-04-25 19:20:25,790 - BERTopic - Topic reduction - Reducing number of topics
2026-04-25 19:20:25,797 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 19:20:25,814 - BERTopic - Representation - Com


📚 Discovered 9 emerging topics:

----------------------------------------------------------------------
  Topic  0: the → to → of → is → in
       (160 articles, 0_the_to_of_is)

  Topic  1: the → to → and → of → for
       (130 articles, 1_the_to_and_of)

  Topic  2: the → fraud → to → of → bbc
       (47 articles, 2_the_fraud_to_of)

  Topic  3: in → attacks → and → elections → groups
       (44 articles, 3_in_attacks_and_elections)

  Topic  4: draft → in → the → series → lakers
       (18 articles, 4_draft_in_the_series)

  Topic  5: ni → flood → india → the → of
       (16 articles, 5_ni_flood_india_the)

  Topic  6: energy → ipo → nuclear → app → day
       (12 articles, 6_energy_ipo_nuclear_app)

  Topic  7: dogs → found → in → been → wolves
       (10 articles, 7_dogs_found_in_been)

  Topic  8: prostate → cancer → he → spread → tumour
       (4 articles, 8_prostate_cancer_he_spread)


📊 Generating BERTopic visualizations...
  ✓ Saved: bertopic_hierarchy.html
  ✓ Saved: bertop